In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [6]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [9]:
q['question']

'How do I join the Office Hours or live workshop if I don’t have the Zoom link?'

In [10]:
import importlib
import rag_helper
importlib.reload(rag_helper)

from rag_helper import RAGBase

In [11]:
answer = assistant.rag(q['question'])

In [12]:
assistant.total_cost()

0.0011205

In [13]:
print(answer)

('The Zoom link is only shared with instructors/presenters/TAs.\n\nIf you’re a student, join via:\n\n- **YouTube Live** — the video URL is usually posted in the **announcements channel on Telegram and Slack** before it starts\n- **Slido** — the link is pinned in the chat when the session is live\n\nYou can also watch on the DataTalksClub **YouTube channel**.', 0)


In [14]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [15]:
assistant.total_cost()

0.0011205

In [16]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': ('The Zoom link is only shared with instructors/presenters/TAs.\n\nIf you’re a student, join via:\n\n- **YouTube Live** — the video URL is usually posted in the **announcements channel on Telegram and Slack** before it starts\n- **Slido** — the link is pinned in the chat when the session is live\n\nYou can also watch on the DataTalksClub **YouTube channel**.',
  0),
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '

In [17]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [18]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': ('The Zoom link is only published to instructors/presenters/TAs.\n\nIf you’re a student, join via YouTube Live instead. The video URL is posted in the announcements channel on Telegram and Slack before the session starts, and the live stream is also available on the DataTalksClub YouTube channel. Questions should be asked in Slido, not in chat.',
  0),
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [19]:
assistant.total_cost()

0.0021869999999999997

In [20]:
assistant.reset_usage()

In [21]:
assistant.total_cost()

0.0

In [22]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [40]:
ground_truth = [
    rec for rec in ground_truth
    if rec["document"] in doc_idx
]

len(ground_truth)

315

In [26]:
with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/315 [00:00<?, ?it/s]

In [27]:
results[:10]

[{'question': 'Is it okay to join the course late if I just found it now?',
  'answer_llm': ('Yes, you can still join the course late if you just discovered it. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
   0),
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'answer_llm': ('Yes — you can start whenever you want. The videos and GitHub materials are available, and you can learn and submit homework while the course platform is still accepting submissions.\n\nIf you want a certificate, make sure to submit your capstone project before submissions close.',
   0),
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {

In [28]:
df_results = pd.DataFrame(results)

In [29]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Is it okay to join the course late if I just f...,"(Yes, you can still join the course late if yo...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take this course even if I missed ...,(Yes — you can start whenever you want. The vi...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,If I join after the course has already started...,(Yes — if you join after the course has starte...,"Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"(Yes — to get the certificate, you need to sub...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,I’m a bit late to the course—what do I need to...,"(If you’re a bit late, you can still join the ...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [30]:
assistant.total_cost()

0.30133949999999987

In [31]:
df_results.to_csv("data/rag-answers-new.csv", index=False)